In [1]:
import pandas as pd

items_df = pd.read_csv("data/grocery_items.csv")
store_df = pd.read_csv("data/store_locations.csv")

print("🛍️ Items Dataset:")
display(items_df)

print("📍 Store Locations Dataset:")
display(store_df)


🛍️ Items Dataset:


,item,price,store
0,Milk,50,D-Mart
1,Milk,55,Reliance Fresh
2,Bread,30,D-Mart
3,Bread,32,Big Bazaar
4,Eggs,60,Big Bazaar
5,Eggs,58,Reliance Fresh
6,Tomato,25,D-Mart
7,Tomato,28,Big Bazaar


📍 Store Locations Dataset:


,store,latitude,longitude
0,D-Mart,19.1210,72.8777
1,Reliance Fresh,19.1190,72.8802
2,Big Bazaar,19.1225,72.8750


In [2]:
grocery_list = ["Milk", "Eggs", "Tomato"]

user_location = (19.1200, 72.8800)


In [3]:
selected_items = []
store_item_map = {}
total_cost = 0

for item in grocery_list:
    item_rows = items_df[items_df["item"].str.lower() == item.lower()]
    
    if item_rows.empty:
        print(f"⚠️ '{item}' not available in any store.")
        continue

    best = item_rows.loc[item_rows["price"].idxmin()]
    
    store = best["store"]
    price = best["price"]

    if store not in store_item_map:
        store_item_map[store] = []
    store_item_map[store].append(item)

    selected_items.append(best)
    total_cost += price

print("✅ Cheapest store selected for each item:\n")
for store, items in store_item_map.items():
    print(f"{store}: {', '.join(items)}")
    
print(f"\n💰 Total Estimated Cost: ₹{total_cost}")


✅ Cheapest store selected for each item:

D-Mart: Milk, Tomato
Reliance Fresh: Eggs

💰 Total Estimated Cost: ₹133


In [4]:
!pip install geopy


In [5]:
from geopy.distance import geodesic

In [6]:
total_distance = 0
store_distance_map = {}

for store in store_item_map:
    store_row = store_df[store_df["store"] == store]
    
    if not store_row.empty:
        lat, lon = store_row.iloc[0]["latitude"], store_row.iloc[0]["longitude"]
        store_coords = (lat, lon)
        
        distance_km = geodesic(user_location, store_coords).km
        store_distance_map[store] = round(distance_km, 2)
        total_distance += distance_km
    else:
        print(f"⚠️ Coordinates not found for {store}")

print("📍 Distances from User to Each Store:\n")
for store, dist in store_distance_map.items():
    print(f"{store}: {dist} km")

print(f"\n🧭 Total Travel Distance: {round(total_distance, 2)} km")


📍 Distances from User to Each Store:

D-Mart: 0.27 km
Reliance Fresh: 0.11 km

🧭 Total Travel Distance: 0.38 km


In [7]:
!pip install folium

In [7]:
import folium

map_center = user_location
grocery_map = folium.Map(location=map_center, zoom_start=15)

folium.Marker(
    location=user_location,
    popup="You are here 🧍",
    icon=folium.Icon(color='blue', icon='user')
).add_to(grocery_map)


In [8]:
for store, items in store_item_map.items():
    store_row = store_df[store_df["store"] == store]
    
    if not store_row.empty:
        lat, lon = store_row.iloc[0]["latitude"], store_row.iloc[0]["longitude"]
        dist = store_distance_map.get(store, 0)
        
        popup_text = f"<b>{store}</b><br>Items: {', '.join(items)}<br>Distance: {dist} km"
        
        folium.Marker(
            location=(lat, lon),
            popup=popup_text,
            icon=folium.Icon(color='green', icon='shopping-cart')
        ).add_to(grocery_map)


In [9]:
grocery_map


In [10]:
import gradio as gr
from geopy.distance import geodesic
import pandas as pd

items_df = pd.read_csv("data/grocery_items.csv")
store_df = pd.read_csv("data/store_locations.csv")

def optimize_grocery_list(grocery_input, latitude, longitude):
    grocery_list = [item.strip() for item in grocery_input.split(",")]
    user_location = (latitude, longitude)

    selected_items = []
    store_item_map = {}
    total_cost = 0

    for item in grocery_list:
        rows = items_df[items_df["item"].str.lower() == item.lower()]
        if rows.empty:
            continue
        best_row = rows.loc[rows["price"].idxmin()]
        store = best_row["store"]
        price = best_row["price"]
        if store not in store_item_map:
            store_item_map[store] = []
        store_item_map[store].append(item)
        selected_items.append(best_row)
        total_cost += price

    total_distance = 0
    store_distance_map = {}
    for store in store_item_map:
        row = store_df[store_df["store"] == store]
        if not row.empty:
            lat, lon = row.iloc[0]["latitude"], row.iloc[0]["longitude"]
            dist = geodesic(user_location, (lat, lon)).km
            store_distance_map[store] = round(dist, 2)
            total_distance += dist

    output = "🛒 **Optimized Grocery Plan**\n\n"
    for store, items in store_item_map.items():
        dist = store_distance_map[store]
        output += f"🏬 **{store}** ({dist} km):\n"
        for item in items:
            price = items_df[(items_df["item"].str.lower() == item.lower()) & (items_df["store"] == store)].iloc[0]["price"]
            output += f"- {item}: ₹{price}\n"
        output += "\n"

    output += f"💰 **Total Cost**: ₹{total_cost}\n"
    output += f"📍 **Total Distance**: {round(total_distance, 2)} km"
    return output


In [11]:
interface = gr.Interface(
    fn=optimize_grocery_list,
    inputs=[
        gr.Textbox(label="Enter Grocery List (comma-separated)", placeholder="Milk, Eggs, Tomato"),
        gr.Number(label="Your Latitude", value=19.1200),
        gr.Number(label="Your Longitude", value=72.8800)
    ],
    outputs=gr.Textbox(label="Optimized Plan"),
    title="Smart Grocery List Optimizer",
    description="Enter your grocery list and location to get the best store combo!"
)

interface.launch()


* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [22]:
!pip install fpdf


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40714 sha256=17b7518d1bd6e183b4137e5164a7180d314868e19076d40952fb17342f1dbaf6
  Stored in directory: c:\users\gauresh\appdata\local\pip\cache\wheels\6e\62\11\dc73d78e40a218ad52e7451f30166e94491be013a7850b5d75
Successfully built fpdf


In [12]:
import pandas as pd
from geopy.distance import geodesic

items_df = pd.read_csv("data/grocery_items.csv")
store_df = pd.read_csv("data/store_locations.csv")

def optimize_grocery_list_export(grocery_input, latitude, longitude):
    grocery_list = [item.strip() for item in grocery_input.split(",")]
    user_location = (latitude, longitude)

    results = []
    total_cost = 0
    total_distance = 0

    for item in grocery_list:
        rows = items_df[items_df["item"].str.lower() == item.lower()]
        if rows.empty:
            continue
        best_row = rows.loc[rows["price"].idxmin()]
        store = best_row["store"]
        price = best_row["price"]

        store_row = store_df[store_df["store"] == store]
        if not store_row.empty:
            lat, lon = store_row.iloc[0]["latitude"], store_row.iloc[0]["longitude"]
            distance = geodesic(user_location, (lat, lon)).km
        else:
            distance = 0

        results.append({
            "Item": item,
            "Store": store,
            "Price (₹)": price,
            "Distance (km)": round(distance, 2)
        })

        total_cost += price
        total_distance += distance

    df = pd.DataFrame(results)
    df.loc[len(df.index)] = ["TOTAL", "", total_cost, round(total_distance, 2)]
    return df


In [13]:
import os
from fpdf import FPDF

def save_as_csv(df):
    path = "data/optimized_list.csv"
    df.to_csv(path, index=False)
    return path

def save_as_pdf(df):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    pdf.cell(200, 10, txt="Smart Grocery List Optimizer", ln=True, align='C')

    for i, row in df.iterrows():
        line = f"{row['Item']} | {row['Store']} | ₹{row['Price (₹)']} | {row['Distance (km)']} km"
        pdf.cell(200, 10, txt=line, ln=True)

    path = "data/optimized_list.pdf"
    pdf.output(path)
    return path


In [14]:
from fpdf import FPDF

def save_as_pdf(df):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    pdf.cell(200, 10, txt="Smart Grocery List Optimizer", ln=True, align='C')
    pdf.ln(5)  

    pdf.set_font("Arial", "B", size=11)
    pdf.cell(50, 10, "Item", border=1)
    pdf.cell(50, 10, "Store", border=1)
    pdf.cell(40, 10, "Price", border=1)
    pdf.cell(40, 10, "Distance", border=1)
    pdf.ln()

    pdf.set_font("Arial", size=11)
    for index in range(len(df)):
        item = str(df.iloc[index]["Item"])
        store = str(df.iloc[index]["Store"])
        price = f"Rs.{df.iloc[index]['Price (₹)']}"
        distance = f"{df.iloc[index]['Distance (km)']} km"

        pdf.cell(50, 10, item, border=1)
        pdf.cell(50, 10, store, border=1)
        pdf.cell(40, 10, price, border=1)
        pdf.cell(40, 10, distance, border=1)
        pdf.ln()

    path = "data/optimized_list.pdf"
    pdf.output(path)
    return path


In [15]:
import gradio as gr

def run_optimizer(grocery_input, lat, lon):
    df = optimize_grocery_list_export(grocery_input, lat, lon)
    csv_path = save_as_csv(df)
    pdf_path = save_as_pdf(df)
    return df, csv_path, pdf_path

gr.Interface(
    fn=run_optimizer,
    inputs=[
        gr.Textbox(label="Enter Grocery List (comma-separated)"),
        gr.Number(label="Latitude", value=19.1200),
        gr.Number(label="Longitude", value=72.8800)
    ],
    outputs=[
        gr.Dataframe(label="Optimized Plan"),
        gr.File(label="📄 Download CSV"),
        gr.File(label="📄 Download PDF")
    ],
    title="Smart Grocery List Optimizer",
    description="Get cheapest store combo and download results as PDF/CSV."
).launch()


* Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


In [16]:
import folium

def generate_folium_map(df, user_location):
    grocery_map = folium.Map(location=user_location, zoom_start=15)

    folium.Marker(
        location=user_location,
        popup="You are here 🧍",
        icon=folium.Icon(color='blue', icon='user')
    ).add_to(grocery_map)

    for _, row in df.iterrows():
        if row["Item"] == "TOTAL": continue  
        store = row["Store"]
        store_row = store_df[store_df["store"] == store]
        if not store_row.empty:
            lat, lon = store_row.iloc[0]["latitude"], store_row.iloc[0]["longitude"]
            popup_text = f"<b>{store}</b><br>Item: {row['Item']}<br>₹{row['Price (₹)']}<br>{row['Distance (km)']} km"
            folium.Marker(
                location=(lat, lon),
                popup=popup_text,
                icon=folium.Icon(color='green', icon='shopping-cart')
            ).add_to(grocery_map)

    return grocery_map._repr_html_()


In [17]:
def run_optimizer_with_map(grocery_input, lat, lon):
    df = optimize_grocery_list_export(grocery_input, lat, lon)
    csv_path = save_as_csv(df)
    pdf_path = save_as_pdf(df)
    html_map = generate_folium_map(df, (lat, lon))
    return df, csv_path, pdf_path, html_map


In [18]:
gr.Interface(
    fn=run_optimizer_with_map,
    inputs=[
        gr.Textbox(label="Enter Grocery List (comma-separated)"),
        gr.Number(label="Latitude", value=19.1200),
        gr.Number(label="Longitude", value=72.8800)
    ],
    outputs=[
        gr.Dataframe(label="Optimized Plan"),
        gr.File(label="📄 Download CSV"),
        gr.File(label="📄 Download PDF"),
        gr.HTML(label="🗺️ Grocery Store Map")
    ],
    title="Smart Grocery List Optimizer",
    description="Get the cheapest store combo, download results, and view stores on a map."
).launch()


* Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.
